# replace-final-head — worked example 2: Replace the Head of a Model Where the Classifier is at model.classifier[-1]

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `replace-final-head`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Not all models use a flat `model.fc` attribute. VGG and AlexNet style models store the classifier as `model.classifier`, which is a `nn.Sequential`. The final head is `model.classifier[-1]`. Replacing it follows the same pattern: read `in_features` from the old layer, then assign a new `nn.Linear` to the right attribute path.

## Worked solution

**Step 1 — locate the final layer.** For a `nn.Sequential` classifier, the last element is `model.classifier[-1]`. Check its type is `nn.Linear` before proceeding.

**Step 2 — read `in_features`.** `model.classifier[-1].in_features` gives the width expected by the final layer. This is the feature dimension output by all the earlier classifier layers.

**Step 3 — replace via index assignment.** `model.classifier[-1] = nn.Linear(in_features, new_num_classes)` replaces the last element in the Sequential. PyTorch updates the module registry automatically.

**Step 4 — verify.** Check `model.classifier[-1].out_features == new_num_classes`.

**Step 5 — test forward pass.** A `(B, input_dim)` input should produce `(B, new_num_classes)` output.

In [ ]:
import torch as t
import torch.nn as nn

class VGGStyleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32, 128), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1000)   # original 1000-class head
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def replace_sequential_head(model: nn.Module, new_num_classes: int) -> nn.Module:
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, new_num_classes)
    return model

# --- exercise and print ---
t.manual_seed(1)
model = VGGStyleModel()
print('Before:', model.classifier[-1].out_features)  # 1000

model = replace_sequential_head(model, new_num_classes=5)
print('After: ', model.classifier[-1].out_features)  # 5
print('in_features preserved:', model.classifier[-1].in_features)  # 128

x = t.randn(3, 20)
out = model(x)
print('output shape:', tuple(out.shape))  # (3, 5)